Our prompt needs to assist users in writing three specific types of output for AWS use cases:

- Python code
- JSON configuration files
- Regular expressions

In [1]:
from typing import Any

from anthropic import Anthropic
from anthropic.types import MessageParam
from dotenv import load_dotenv

load_dotenv()

Messages = list[MessageParam]

client: Anthropic = Anthropic()
model: str = "claude-sonnet-4-6"
system_prompt = "Respond only with raw valid JSON. No explanation, no markdown, no code blocks."

In [2]:
def add_user_message(messages: Messages, text: str) -> None:
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: Messages, text: str) -> None:
    messages.append({"role": "assistant", "content": text})

def chat(messages: Messages, system_prompt: str|None = None, temperature: float = 1.0) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system_prompt is not None:
        params["system"] = system_prompt
    
    response = client.messages.create(**params)
    return response.content[0].text

In [3]:
import json

def generate_dataset() -> list[dict[str, str]]:
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages: Messages = []
    add_user_message(messages, prompt)
    text = chat(messages, system_prompt=system_prompt)
    return json.loads(text.strip())

In [4]:
dataset = generate_dataset()

with open("dataset.local.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [5]:
def run_prompt(test_case: dict[str, str]) -> str:
    """Merges the prompt and test case input, then returns the result."""
    
    prompt = f""""
Please, solve the following task:

{test_case["task"]}
    """
    messages: Messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [6]:
def grade_by_model(task: str, output: str) -> dict[str, Any]:
    eval_prompt = f"""You are an expert code reviewer. Evaluate this AI-generated solution.

Original task:
<task>
{task}
</task>

Solution to evaluate:
<solution>
{output}
</solution>

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement  
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10

Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
"""

    messages = []
    add_user_message(messages, eval_prompt)

    eval_text = chat(messages, system_prompt=system_prompt)
    return json.loads(eval_text)

In [7]:
def run_test_case(test_case: dict[str, str]) -> dict[str, Any]:
    """Calls run_prompt, then grades the result."""
    
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case["task"], output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [ ]:
from statistics import mean

def run_eval(dataset: list[dict[str, str]]) -> list[dict[str, Any]]:
    """Loads the dataset and calls run_test_case with each case."""

    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [9]:
with open("dataset.local.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# S3 Object Keys Retrieval Function\n\n## Solution\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError, NoCredentialsError\nfrom typing import Optional\n\n\ndef list_s3_objects(bucket_name: str, prefix: str = \"\") -> list[str]:\n    \"\"\"\n    List all object keys in an S3 bucket matching the given prefix.\n\n    Args:\n        bucket_name: The name of the S3 bucket.\n        prefix: The prefix to filter objects (default: \"\" for all objects).\n\n    Returns:\n        A list of object keys matching the given prefix.\n\n    Raises:\n        NoCredentialsError: If AWS credentials are not configured.\n        ClientError: If there's an AWS service error (e.g., bucket not found).\n        ValueError: If bucket_name is empty or None.\n    \"\"\"\n    if not bucket_name:\n        raise ValueError(\"Bucket name cannot be empty or None.\")\n\n    s3_client = boto3.client(\"s3\")\n    object_keys = []\n\n    try:\n        # Use a paginator to handle b